<a href="https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:

## Baseline Rule

'''This baseline prioritizes pages that receive a high number of impressions but relatively few clicks.

The assumption is that pages receiving visibility in search results but generating limited traffic may represent SEO opportunities.

Pages are ranked using an Opportunity Score.

Higher scores indicate a potentially larger opportunity for optimization.

### Action Label

SEO_OPPORTUNITY

### Reason Codes

HIGH_IMPRESSIONS_LOW_CLICKS

HIGH_IMPRESSIONS_LOW_CTR

HIGH_VISIBILITY_UNDERPERFORMING'''

'This baseline prioritizes pages that receive a high number of impressions but relatively few clicks.\n\nThe assumption is that pages receiving visibility in search results but generating limited traffic may represent SEO opportunities.\n\nPages are ranked using an Opportunity Score.\n\nHigher scores indicate a potentially larger opportunity for optimization.\n\n### Action Label\n\nSEO_OPPORTUNITY\n\n### Reason Codes\n\nHIGH_IMPRESSIONS_LOW_CLICKS\n\nHIGH_IMPRESSIONS_LOW_CTR\n\nHIGH_VISIBILITY_UNDERPERFORMING'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(march_file)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [ ]:
queue = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('{march_file}')
WHERE gsc_data_available IS TRUE
""").df()

print(queue.shape)

queue.head()

(3611061, 6)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727


In [ ]:
queue["ctr"] = queue["gsc_clicks"] / queue["gsc_impressions"]

queue["ctr"] = queue["ctr"].fillna(0)

queue["score"] = (
    queue["gsc_impressions"] *
    (1 - queue["ctr"])
)

queue["reason_code"] = "HIGH_IMPRESSIONS_LOW_CTR"

queue["action"] = "SEO_OPPORTUNITY"

queue = queue.sort_values(
    by="score",
    ascending=False
)

queue.head(20)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
3534021,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.083350,0.000025,40083.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY
2949804,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,2.197507,0.006411,39053.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY
43030,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,2.764916,0.000051,39001.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY
3358182,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,2.195988,0.007051,38165.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY
43077,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,0.000000,37368.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY
3274261,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,2.188397,0.006355,35179.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY
3169375,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,223,2.181348,0.006405,34594.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY
3578638,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,2.242501,0.006791,34371.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY
3248019,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.181500,0.000000,33383.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY
2677007,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,33571,215,2.309046,0.006404,33356.0,HIGH_IMPRESSIONS_LOW_CTR,SEO_OPPORTUNITY


In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV Saved Successfully")

CSV Saved Successfully


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
### Rank 1

'''Action:
SEO_OPPORTUNITY

Reason Code:
HIGH_IMPRESSIONS_LOW_CTR

Confidence:
High

What would make it wrong?
The page may already be optimized and low CTR could be caused by SERP features.

---

### Rank 2

Action:
SEO_OPPORTUNITY

Reason Code:
HIGH_IMPRESSIONS_LOW_CTR

Confidence:
High

What would make it wrong?
The page may attract informational queries that naturally receive lower CTR.

---

### Rank 3

Action:
SEO_OPPORTUNITY

Reason Code:
HIGH_IMPRESSIONS_LOW_CTR

Confidence:
Medium

What would make it wrong?
Seasonal traffic patterns may temporarily influence impressions and clicks.

---

### Rank 4

Action:
SEO_OPPORTUNITY

Reason Code:
HIGH_IMPRESSIONS_LOW_CTR

Confidence:
Medium

What would make it wrong?
Recent page changes may not yet be reflected in performance data.

---

### Rank 5

Action:
SEO_OPPORTUNITY

Reason Code:
HIGH_IMPRESSIONS_LOW_CTR

Confidence:
Medium

What would make it wrong?
The page may already be ranking at its realistic ceiling.'''

'Action:\nSEO_OPPORTUNITY\n\nReason Code:\nHIGH_IMPRESSIONS_LOW_CTR\n\nConfidence:\nHigh\n\nWhat would make it wrong?\nThe page may already be optimized and low CTR could be caused by SERP features.\n\n---\n\n### Rank 2\n\nAction:\nSEO_OPPORTUNITY\n\nReason Code:\nHIGH_IMPRESSIONS_LOW_CTR\n\nConfidence:\nHigh\n\nWhat would make it wrong?\nThe page may attract informational queries that naturally receive lower CTR.\n\n---\n\n### Rank 3\n\nAction:\nSEO_OPPORTUNITY\n\nReason Code:\nHIGH_IMPRESSIONS_LOW_CTR\n\nConfidence:\nMedium\n\nWhat would make it wrong?\nSeasonal traffic patterns may temporarily influence impressions and clicks.\n\n---\n\n### Rank 4\n\nAction:\nSEO_OPPORTUNITY\n\nReason Code:\nHIGH_IMPRESSIONS_LOW_CTR\n\nConfidence:\nMedium\n\nWhat would make it wrong?\nRecent page changes may not yet be reflected in performance data.\n\n---\n\n### Rank 5\n\nAction:\nSEO_OPPORTUNITY\n\nReason Code:\nHIGH_IMPRESSIONS_LOW_CTR\n\nConfidence:\nMedium\n\nWhat would make it wrong?\nThe page

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
## Weak Picks

'''Some high-scoring pages may not represent genuine opportunities.

Possible weak picks include:

- Pages affected by SERP features.
- Seasonal content.
- Recently updated pages.
- Queries with naturally low CTR.

These pages may receive high scores even though limited optimization opportunity exists.

## Leakage Check

The baseline score only uses:

- gsc_impressions
- gsc_clicks
- gsc_avg_position

No future information was used.

No label-derived fields were used.

No client identifiers or product access flags were used as predictive signals.

The score relies only on information available before the recommendation decision.'''

'Some high-scoring pages may not represent genuine opportunities.\n\nPossible weak picks include:\n\n- Pages affected by SERP features.\n- Seasonal content.\n- Recently updated pages.\n- Queries with naturally low CTR.\n\nThese pages may receive high scores even though limited optimization opportunity exists.\n\n## Leakage Check\n\nThe baseline score only uses:\n\n- gsc_impressions\n- gsc_clicks\n- gsc_avg_position\n\nNo future information was used.\n\nNo label-derived fields were used.\n\nNo client identifiers or product access flags were used as predictive signals.\n\nThe score relies only on information available before the recommendation decision.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.